In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w8" / "day04"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
from app.tools import registry
from app.tools import get_document_versions

t = registry.get('get_document_versions')
print('등급 :', t.side_effect.value)

out = t.run(doc_id='DOC-HR-014')

print(f"{out['title']} · {out['dept']} · {out['security_level']}")
for v in out['versions']:
    print(f"{v['version']} · {v['status']} · {v['period']} · 청크 {v['chunk_count']} · 검색 {v['searchable']}")

등급 : none
국내출장 여비 규정 · 인사총무 · 일반
v2.0 · 현행 · 2025-07-01 ~ · 청크 53 · 검색 False
v1.1 · 만료 · 2024-09-01 ~ 2025-06-30 · 청크 24 · 검색 False


In [3]:
import math
from app.services import chat_service
text = (ROOT / 'backend/app/services/chat_service.py').read_text(encoding='utf-8')
loops = [line for line in text.splitlines() if 'for turn in range(' in line]
print(f'루프(for turn in range) : {len(loops)}개')
limit = chat_service.MAX_TOOL_TURNS
print(f'MAX_TOOL_TURNS : {limit}')
print(f'등록기 도구 : {len(registry.all_tools())}개')
calls_per_turn = 1
needed_tools = 4
needed_turns = math.ceil(needed_tools / calls_per_turn) + 1
count = {1: '하나를', 2: '둘을', 3: '셋을', 4: '넷을'}.get(needed_tools, f'{needed_tools}개를')
head = f'한 왕복에 {calls_per_turn}개씩 {count} 부르면 필요한 왕복 : {needed_turns}'
if needed_turns > limit:
    print(f'{head} (> {limit} — 최종 답 자리가 없다)')
    print(f'그때 남는 문구 : {chat_service.TOOL_LIMIT_ANSWER}')
else:
    print(f'{head} (<= {limit} — 상한 안에 든다)')


루프(for turn in range) : 1개
MAX_TOOL_TURNS : 4
등록기 도구 : 1개
한 왕복에 1개씩 넷을 부르면 필요한 왕복 : 5 (> 4 — 최종 답 자리가 없다)
그때 남는 문구 : (도구 호출 상한에 닿아 답을 마치지 못했습니다)


In [4]:
from app.services import chat_service

QUESTION = [
    ("질문 1", "국내 출장 여비 규정은 지금 몇 버전 인가요?"),
    ("질문 2", "점심 메뉴 추천해 줘.")
]

for label, q in QUESTION:
    out = chat_service.run_graph(question=q)
    branch = "found" if out ["passed"] else "insufficient"
    pair = ("note" in out) == (not out["passed"])
    print(f"{label} -> 갈래 {branch} - note {'있음' if 'note' in out else '없음'} - 짝이 맞는지 : {pair}") 

14:54:51 INFO     sentence_transformers.base.model : No device provided, using mps
14:54:51 INFO     sentence_transformers.base.model : Loading SentenceTransformer model from ../models/bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

14:54:54 INFO     app.integrations.local_embed : 로컬 임베딩 모델 준비: ../models/bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

14:54:54 INFO     app.rag.retriever : 정책으로 제외: 국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/법인카드 사용 지침 v1.5 - 만료 버전
14:54:54 INFO     sentence_transformers.base.model : No device provided, using mps
14:54:54 INFO     sentence_transformers.base.model : No modules.json found for ../models/bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

14:54:56 INFO     app.integrations.local_rerank : 로컬 리랭커 준비 : ../models/bge-reranker-v2-m3, 최대 512토큰
질문 1 -> 갈래 found - note 없음 - 짝이 맞는지 : True


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

14:54:57 INFO     app.rag.retriever : 정책으로 제외: 국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/법인카드 사용 지침 v2.0 - 만료 버전
질문 2 -> 갈래 insufficient - note 있음 - 짝이 맞는지 : True
